# A2UI with LangChain

A simple chain where Claude answers with **A2UI** messages (UI described as JSON).

```
question → prompt → Claude → to_a2ui (parse) → A2UI messages
```

In [1]:
import json
import subprocess

from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

from a2ui.core.basic_catalog import BasicCatalog
from a2ui.inference_formats.direct_json import DirectJsonFormat
from a2ui.schema import VERSION_0_9, CatalogConfig

## 1. Catalog
The components the client is allowed to render (Card, Text, Button, ...).

In [2]:
catalog = CatalogConfig.from_catalog("basic", BasicCatalog(VERSION_0_9)).to_catalog(
    protocol_version=VERSION_0_9
)
a2ui = DirectJsonFormat([catalog])  # the format the model writes: plain A2UI JSON

## 2. Prompt
The SDK writes a system prompt that teaches the model the A2UI schema and catalog.

In [ ]:
# turns the format rules and catalog into system instructions for model
system_prompt = a2ui.generate_system_prompt(
    role_description="Answer with one short sentence, then an A2UI UI.",
    include_schema=True,
)

# SystemMessage is an object, not a template, so the { } in the schema are left alone.
prompt = ChatPromptTemplate.from_messages(
    [SystemMessage(system_prompt), ("human", "{question}")]
)

print(system_prompt[:500], "...")

Answer with one short sentence, then an A2UI UI.

## Workflow Description:

The generated response MUST follow these rules:
- The response can contain one or more A2UI JSON blocks.
- Each A2UI JSON block MUST be wrapped in `<a2ui-json>` and `</a2ui-json>` tags.
- Between or around these blocks, you can provide conversational text.
- The JSON part MUST be a single, raw JSON object (usually a list of A2UI messages) and MUST validate against the provided A2UI JSON SCHEMA.
- Top-Down Component Order ...


/var/folders/k2/swnr3fc52yjc83gxdxqzrt6c0000gn/T/ipykernel_5210/3492731952.py:1: DeprecationWarning: generate_system_prompt is deprecated. Use prompt_generator.generate(...) instead.
  system_prompt = a2ui.generate_system_prompt(


## 3. Model
Claude via your Claude Code login (`claude -p`), so no API key is needed.

In [4]:
def claude(prompt_value) -> str:
    system, human = prompt_value.to_messages()
    cmd = ["claude", "-p", "--model", "opus", "--tools", "", "--system-prompt", system.content]
    return subprocess.run(cmd, input=human.content, capture_output=True, text=True, check=True).stdout


llm = RunnableLambda(claude)

## 4. Parser
Pull the A2UI messages out of the reply.

In [5]:
def to_a2ui(reply: str) -> list[dict]:
    parts = a2ui.parser.parse_response(reply)  # splits text from <a2ui-json> blocks
    return [m for part in parts for m in part.a2ui_json or []]


parser = RunnableLambda(to_a2ui)

## 5. Chain

In [6]:
chain = (prompt | llm | parser)

## 6. Run it

In [20]:
messages = chain.invoke({"question": "a weather card for Cairo, 34°C and sunny"})

print(json.dumps(messages, indent=2))

[
  {
    "version": "v0.9",
    "createSurface": {
      "surfaceId": "weather-cairo",
      "catalogId": "https://a2ui.org/specification/v0_9/catalogs/basic/catalog.json",
      "theme": {
        "primaryColor": "#F5A623",
        "agentDisplayName": "Weather"
      }
    }
  },
  {
    "version": "v0.9",
    "updateComponents": {
      "surfaceId": "weather-cairo",
      "components": [
        {
          "id": "root",
          "component": "Card",
          "child": "main-col"
        },
        {
          "id": "main-col",
          "component": "Column",
          "align": "stretch",
          "children": [
            "header-row",
            "divider",
            "temp-row",
            "condition-text"
          ]
        },
        {
          "id": "header-row",
          "component": "Row",
          "justify": "spaceBetween",
          "align": "center",
          "children": [
            "location-row",
            "date-text"
          ]
        },
        {
     

## 7. Save for the React app
Then click **Reload** in the viewer (`cd web && npm run dev`).

In [21]:
with open("web/public/output.jsonl", "w") as f:
    f.writelines(json.dumps(m) + "\n" for m in messages)